In [22]:
from datetime import datetime, timedelta, time

INICIO_EXPEDIENTE = time(8, 0)
FIM_EXPEDIENTE = time(18, 0)

# Lista opcional de feriados
FERIADOS = {
    # datetime(2026, 1, 1).date(),
    # datetime(2026, 4, 3).date(),
}


def eh_dia_util(data):
    return data.weekday() < 5 and data not in FERIADOS


def proximo_dia_util(data):
    data += timedelta(days=1)

    while not eh_dia_util(data.date()):
        data += timedelta(days=1)

    return datetime.combine(data.date(), INICIO_EXPEDIENTE)


def ajustar_para_horario_util(data):
    """
    Ajusta a data para o próximo horário útil válido.
    """

    while not eh_dia_util(data.date()):
        data = proximo_dia_util(data)

    if data.time() < INICIO_EXPEDIENTE:
        return datetime.combine(data.date(), INICIO_EXPEDIENTE)

    if data.time() >= FIM_EXPEDIENTE:
        return proximo_dia_util(data)

    return data


def calcular_vencimento(data_abertura, sla_horas):
    data_atual = ajustar_para_horario_util(data_abertura)
    horas_restantes = sla_horas

    while horas_restantes > 0:

        fim_dia = datetime.combine(
            data_atual.date(),
            FIM_EXPEDIENTE
        )

        horas_disponiveis = (
            fim_dia - data_atual
        ).total_seconds() / 3600

        if horas_restantes <= horas_disponiveis:
            return data_atual + timedelta(hours=horas_restantes)

        horas_restantes -= horas_disponiveis
        data_atual = proximo_dia_util(data_atual)

    return data_atual


In [23]:
ano = 2026
mes= 11
dia = 7
hora = 16
minuto = 15


abertura = datetime(ano, mes, dia, hora, minuto)

vencimento = calcular_vencimento(abertura,20)

print(vencimento.strftime("%d/%m/%Y %H:%M"))

10/11/2026 18:00
